# 060 — Site case-study MSA collapse fragilities

Collects the Multiple-Stripe-Analysis (MSA) collapse fragility of each case-study
structure from the analysis drive, copies it into the processed-data tree **with a
provenance manifest**, plots it against its stripe IMLs, and reports which curves are
*not well defined* — i.e. whose stripes never bracket collapse.

The processed copies written here are what notebook **061** consumes.

## Inputs

- `D:/07_wp1_casestudy_sites/site_{i}/{n}s/mdof/msa_AvgSA_03/collapse_fragility.json` —
  the MSA run output (`median`, `dispersion`, and the 2xN empirical curve `efc`).
- `data_processed/05_gcim_distributions/imls_for_selection_AvgSA_03.json` — the stripe
  IMLs, drawn on the plots as guide-lines.

## Outputs

| File | Content |
| --- | --- |
| `data_processed/09_structure_fragility_curves/wp1_casestudy_sites/site_{i}/{tag}_msa_collapsefragility_AvgSA_03.json` | the copied fragility (read by nb 061) |
| `.../{tag}_msa_collapsefragility_AvgSA_03.json.manifest.json` | its provenance sidecar |
| `results/05_site_fragility_curves/fragility_curves_site_{i}_{n}s.jpg` | the fragility plot |

## Caching

Each structure is cached against the **content hash of its source
`collapse_fragility.json`**, via `cache_utils.json_load_or_compute` — the analysed IMLs
live inside that file as `efc[0]`, so its hash already pins which stripes the curve was
fitted to. A structure whose `.manifest.json` still matches is
left untouched — the JSON is not rewritten and the figure is not redrawn (unless the JPG
has gone missing). A re-run after re-running a handful of MSA analyses therefore
reprocesses only those. Set `FORCE_RECOMPUTE = True` to rebuild everything.

Section 5 then decides what to do about the curves whose stripes failed to bracket
collapse, and writes the two remediation lists that notebooks 017 and 050 read back.

> **Run order.** The MSA results live on `D:` — run this on the machine with the drive
> attached to (re)build the processed copies. Without it the notebook prints a warning
> and falls back to the copies already in `data_processed/09_.../`, so the plots and the
> well-definedness report still work, but nothing is written.

In [1]:
%load_ext autoreload
%autoreload 2

## 0. Setup & parameters

In [2]:
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import lognorm

from phd_project.config import config
from phd_project.scripts.cache_utils import fingerprint, json_load_or_compute

cfg = config.load_config()

In [3]:
# ----------------------------------------------------------------------------
# PARAMETERS
# ----------------------------------------------------------------------------
GM_SET = "AvgSA_03"
N_STOREYS = [3, 5]              # structure types to process
SITES = list(range(0, 60))      # case-study site indices

# --- FOLDERS ---
ANALYSIS_ROOT = cfg["analysis_data"]["wp1_casestudy_sites"]   # D: - the MSA run output
FRAG_ROOT = cfg["proc_data"]["wp1_sites_fragility_curves"]    # the processed JSON copies
RESULTS_ROOT = cfg["results"]["site_fragility_curves"]        # the fragility JPGs
STRIPE_IML_PATH = cfg["proc_data"][f"{GM_SET}_imls_for_selection"]

RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

# --- WELL-DEFINEDNESS THRESHOLDS ---
# A curve is well defined only if its stripes bracket collapse: the empirical curve must
# rise ABOVE PC_UPPER_THRESHOLD and start BELOW PC_LOWER_THRESHOLD.
PC_UPPER_THRESHOLD = 0.8
PC_LOWER_THRESHOLD = 0.2

# --- REMEDIATION BANDS ---
# Where a *remediating* stripe must land, tighter than the pass/fail thresholds above:
# 2-5 and 25-28 collapses out of the 30 records per stripe, i.e. far enough from an
# all-collapse / no-collapse stripe that one more MSA round is definitely enough.
PC_BAND_LOW = (0.067, 0.167)
PC_BAND_HIGH = (0.833, 0.933)

# Every IML in this pipeline is stored rounded to 4 dp, while the band edges below are
# computed from a continuous lognormal. Compare the two with half an ulp of slack, or an
# IML already on the grid gets missed by a hair and is reported as needing a new
# disaggregation it does not need.
IML_TOL = 5e-5

DISAGG_SIGMA = 4    # truncation level nb 017 picked the stripe IMLs at
DISAGG_GRID_PATH = cfg["proc_data"][f"disagg_imls_{GM_SET}"]
HAZARD_CURVE_PATH = cfg["proc_data"][f"{GM_SET}_hazard_curves_{DISAGG_SIGMA}sig"]

# --- REMEDIATION OUTPUTS (written by section 5) ---
# Both are read back by nb 017 (additional IMLs -> the disaggregation grid and the per-site
# record-selection lists) and nb 050 (reserve stripes -> the MSA run list).
RESERVE_STRIPE_PATH = cfg["proc_data"][f"{GM_SET}_reserve_stripes"]
ADDITIONAL_IML_PATH = cfg["proc_data"][f"{GM_SET}_additional_imls"]

# Escape hatch: re-copy and re-plot every structure, ignoring the per-structure
# manifests. Leave False for normal incremental runs (only new/changed MSA runs rebuild).
FORCE_RECOMPUTE = False


def structure_tag(site_idx: int, n: int) -> str:
    return f"{n}s_cbf_dc2_site{site_idx}"


print(f"ANALYSIS_ROOT = {ANALYSIS_ROOT}")
print(f"FRAG_ROOT     = {FRAG_ROOT}")
print(f"RESULTS_ROOT  = {RESULTS_ROOT}")

ANALYSIS_ROOT = D:\07_wp1_casestudy_sites
FRAG_ROOT     = C:\Users\clemettn\Documents\phd\data_processed\09_structure_fragility_curves\wp1_casestudy_sites
RESULTS_ROOT  = C:\Users\clemettn\Documents\phd\results\05_site_fragility_curves


## 1. Stripe IMLs

The IMLs each structure's MSA stripes were run at, keyed by site index then structure
tag. They are drawn on the fragility plots as guide-lines and folded into the cache
fingerprint, so editing the IML file redraws the affected figures.

In [4]:
with open(STRIPE_IML_PATH, "r") as file:
    stripe_imls = json.load(file)

print(f"{len(stripe_imls)} sites in {STRIPE_IML_PATH.name}")

60 sites in imls_for_selection_AvgSA_03.json


## 2. Analysis-root availability

`ANALYSIS_ROOT` lives on the external `D:` drive. When it is not attached this is a
**warning, not an error**: the notebook switches to reading the already-processed copies
under `FRAG_ROOT` so the plots and the report below still run, and writes nothing.

In [5]:
ANALYSIS_OK = ANALYSIS_ROOT.exists()

if ANALYSIS_OK:
    print(f"Analysis root available: {ANALYSIS_ROOT}")
else:
    print(f"WARNING: analysis root {ANALYSIS_ROOT} is not accessible (drive not attached).")
    print(f"  Falling back to the processed copies in {FRAG_ROOT}.")
    print("  No fragility JSONs will be (re)written.")

Analysis root available: D:\07_wp1_casestudy_sites


## 3. Process the MSA fragility curves

For every `(site, storeys)`: resolve the source fragility on the analysis drive, pass it
through the provenance cache, run the well-definedness check, and redraw the figure only
when the cache was rebuilt (or the JPG is missing).

Statuses reported at the end:

- **computed** — source changed (or is new); JSON + manifest + JPG (re)written.
- **cached** — manifest matched; nothing rewritten.
- **offline** — read from the processed copy because `D:` is detached; nothing written.
- **skipped** — no MSA fragility available for that structure at all.

In [6]:
def fmt_sites(sites: list[int]) -> str:
    """Compact, sorted site listing for the printed summaries."""
    return ", ".join(str(s) for s in sorted(sites)) if sites else "-"


def msa_source_path(site: int, ns: int) -> Path:
    """The collapse fragility written by the MSA run, on the analysis drive."""
    return (ANALYSIS_ROOT / f"site_{site}" / f"{ns}s" / "mdof"
            / f"msa_{GM_SET}" / "collapse_fragility.json")


def processed_path(site: int, ns: int) -> Path:
    """The processed copy consumed by notebook 061."""
    tag = structure_tag(site, ns)
    return FRAG_ROOT / f"site_{site}" / f"{tag}_msa_collapsefragility_{GM_SET}.json"


def plot_path(site: int, ns: int) -> Path:
    return RESULTS_ROOT / f"fragility_curves_site_{site}_{ns}s.jpg"


def plot_fragility_curve(fc, site, ns, out_fp):
    """Fitted lognormal + empirical MSA points, with the analysed stripe IMLs marked.

    The guide-lines come from the empirical curve itself, so they mark the stripes that
    were actually run rather than the wider candidate list in imls_for_selection.
    """
    im_max = lognorm.ppf(0.99, s=fc["dispersion"], scale=fc["median"])
    imls = np.linspace(0, im_max * 1.15, 50)   # in g
    msa_fc_fit = lognorm.cdf(imls, s=fc["dispersion"], scale=fc["median"])

    fig, ax = plt.subplots(figsize=(6, 4))
    plt.close(fig)

    for siml in fc["efc"][0, :]:
        ax.axvline(siml, ls="--", color="k", alpha=0.5)

    ax.plot(imls, msa_fc_fit, color="b", label="MSA")
    ax.plot(fc["efc"][0, :], fc["efc"][1, :], ls="none", marker="o", mfc="b", mec="k",
            alpha=0.75, label="MSA ecdf")

    ax.set_ylim(0, 1)
    ax.grid(ls="-.", color="0.8")
    ax.set_xlabel("AvgSA[0,3], [g]")
    ax.set_ylabel("Probability of Collapse, P[C]")
    ax.set_title(f"Fragility Curves - Site {site}, {ns}s")
    leg = ax.legend()
    leg.get_frame().set_edgecolor("k")

    fig.savefig(out_fp, dpi=300, bbox_inches="tight")

In [7]:
not_well_defined = {ns: {"lt_upper_threshold": [], "gt_lower_threshold": []}
                    for ns in N_STOREYS}
assessed = {ns: [] for ns in N_STOREYS}
statuses = {"computed": [], "cached": [], "offline": [], "skipped": []}

for site in SITES:
    for ns in N_STOREYS:
        save_fp = processed_path(site, ns)
        jpg_fp = plot_path(site, ns)

        if ANALYSIS_OK:
            src = msa_source_path(site, ns)
            if not src.is_file():
                statuses["skipped"].append((site, ns))
                continue

            # The cache key is the content of the source fragility, and nothing else. The
            # analysed IMLs are inside it as efc[0], so the hash already pins exactly which
            # stripes this curve was fitted to - whereas the candidate list in
            # imls_for_selection also names stripes that were never run, and grows when
            # nb 017 merges remediation IMLs back in. Re-running the MSA is what
            # invalidates the processed copy, which is what the manifest should claim.
            save_fp.parent.mkdir(parents=True, exist_ok=True)
            fc_raw, status = json_load_or_compute(
                save_fp,
                fingerprint(msa_fragility=src),
                lambda src=src: json.load(open(src)),
                force=FORCE_RECOMPUTE,
                input_paths={"msa_fragility": src},
            )
        else:
            # Offline: reuse the processed copy as-is. Nothing is written and no manifest
            # is touched, so provenance stays whatever the last online run stamped.
            if not save_fp.is_file():
                statuses["skipped"].append((site, ns))
                continue
            with open(save_fp, "r") as file:
                fc_raw = json.load(file)
            status = "offline"

        statuses[status].append((site, ns))
        fc = {k: np.array(v) if isinstance(v, list) else v for k, v in fc_raw.items()}

        # --- well-definedness check: always runs, cached or not, so the report below is
        # complete on every run rather than only covering the rebuilt structures.
        assessed[ns].append(site)
        if max(fc["efc"][1, :]) <= PC_UPPER_THRESHOLD:
            not_well_defined[ns]["lt_upper_threshold"].append(site)
        if min(fc["efc"][1, :]) >= PC_LOWER_THRESHOLD:
            not_well_defined[ns]["gt_lower_threshold"].append(site)

        # --- plot only when the cache was rebuilt, or the figure has gone missing
        if status == "computed" or not jpg_fp.is_file():
            plot_fragility_curve(fc, site, ns, jpg_fp)

print("\n" + "   ".join(f"[{name}] {len(v)}" for name, v in statuses.items()))
if statuses["skipped"]:
    print("  skipped (no MSA fragility available):")
    for ns in N_STOREYS:
        sk = [s for s, n in statuses["skipped"] if n == ns]
        if sk:
            listing = "all sites" if len(sk) == len(SITES) else fmt_sites(sk)
            print(f"    {ns}s - {len(sk)}: {listing}")

[stale] 3s_cbf_dc2_site0_msa_collapsefragility_AvgSA_03.json: recomputing -- changed input: input 'stripe_imls' (no file path)
[stale] 5s_cbf_dc2_site0_msa_collapsefragility_AvgSA_03.json: recomputing -- changed input: input 'stripe_imls' (no file path)
[stale] 3s_cbf_dc2_site1_msa_collapsefragility_AvgSA_03.json: recomputing -- changed input: input 'stripe_imls' (no file path)
[stale] 5s_cbf_dc2_site1_msa_collapsefragility_AvgSA_03.json: recomputing -- changed input: input 'stripe_imls' (no file path)
[stale] 3s_cbf_dc2_site2_msa_collapsefragility_AvgSA_03.json: recomputing -- changed input: input 'stripe_imls' (no file path)
[stale] 5s_cbf_dc2_site2_msa_collapsefragility_AvgSA_03.json: recomputing -- changed input: input 'stripe_imls' (no file path)
[stale] 3s_cbf_dc2_site3_msa_collapsefragility_AvgSA_03.json: recomputing -- changed input: input 'stripe_imls' (no file path)
[stale] 5s_cbf_dc2_site3_msa_collapsefragility_AvgSA_03.json: recomputing -- changed input: input 'stripe_imls'

## 4. Fragility-curve definition check

Which structures the MSA stripes failed to bracket collapse for, **reported separately
for each storey count** — the two structure types have different periods and base-shear
coefficients, so their stripe placement fails in different ways and the counts are only
meaningful when kept apart.

In [8]:
print("Fragility-curve definition check")
print(f"  thresholds: max P[C] must exceed {PC_UPPER_THRESHOLD:.2f}, "
      f"min P[C] must fall below {PC_LOWER_THRESHOLD:.2f}")

for ns in N_STOREYS:
    n_assessed = len(assessed[ns])
    print(f"\n{ns}s structures - {n_assessed} assessed")

    if n_assessed == 0:
        print("  no MSA fragility curves found.")
        continue

    lt = not_well_defined[ns]["lt_upper_threshold"]
    gt = not_well_defined[ns]["gt_lower_threshold"]

    print(f"  max P[C] <= {PC_UPPER_THRESHOLD:.2f} (upper tail not reached) - {len(lt)} sites:")
    print(f"    {fmt_sites(lt)}")
    print(f"  min P[C] >= {PC_LOWER_THRESHOLD:.2f} (lower tail not reached) - {len(gt)} sites:")
    print(f"    {fmt_sites(gt)}")

    n_bad = len(set(lt) | set(gt))
    print(f"  well defined: {n_assessed - n_bad}/{n_assessed}  "
          f"({n_bad} site(s) fail at least one threshold)")

Fragility-curve definition check
  thresholds: max P[C] must exceed 0.80, min P[C] must fall below 0.20

3s structures - 57 assessed
  max P[C] <= 0.80 (upper tail not reached) - 39 sites:
    0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 19, 25, 26, 27, 29, 30, 32, 33, 34, 35, 36, 37, 39, 40, 42, 43, 48, 50, 52, 53, 54, 59
  min P[C] >= 0.20 (lower tail not reached) - 12 sites:
    4, 12, 17, 22, 23, 27, 28, 37, 44, 49, 57, 58
  well defined: 10/57  (47 site(s) fail at least one threshold)

5s structures - 57 assessed
  max P[C] <= 0.80 (upper tail not reached) - 43 sites:
    0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 23, 25, 26, 27, 28, 29, 30, 34, 37, 39, 40, 42, 43, 44, 45, 46, 47, 48, 49, 51, 53
  min P[C] >= 0.20 (lower tail not reached) - 16 sites:
    4, 12, 17, 18, 22, 23, 27, 28, 32, 33, 35, 36, 55, 57, 58, 59
  well defined: 5/57  (52 site(s) fail at least one threshold)


## 5. Stripe-coverage remediation

Section 4 says *which* structures are badly conditioned; this section says *what to do
about each one*, and separates the cheap fix from the expensive one.

**The goal.** Every structure needs at least one stripe below `PC_LOWER_THRESHOLD` and one
above `PC_UPPER_THRESHOLD`, so the MLE fit is anchored on both tails.

**The two remedies.**

1. **Run a reserve stripe** — nb 017 reserved one extra grid IML below and one above the
   four analysed stripes, as each structure's `reserves: [low, high]` field in
   `imls_for_selection_AvgSA_03.json` (either is `null` where the hazard ceiling stopped
   it). Ground-motion ensembles already exist for these IMLs, so this costs an MSA run and
   nothing else — no disaggregation, no record selection.
2. **Add a new IML** — a fresh disaggregation (unless the IML happens to land on the
   existing 13-point disagg grid) plus GCIM and record selection, then the MSA run.

**The acceptance band.** This has to be the last remediation round, so a remediating
stripe is only accepted if its *predicted* P[C] lands in `PC_BAND_LOW` / `PC_BAND_HIGH` —
2-5 or 25-28 collapses of 30. That is deliberately stricter than the 0.2 / 0.8 pass-fail
test: a stripe predicted at 0.02 would technically clear the lower threshold but has a
real chance of coming back with zero collapses.

P[C] at a not-yet-run IML is predicted from the **lognormal already fitted to the four
analysed stripes** (`median`, `dispersion` in each `collapse_fragility.json`).

### Outputs

| File | Content |
| --- | --- |
| `RESERVE_STRIPE_PATH` | `{"3": [[site, iml], ...], "5": [...]}` - reserve stripes to run against the existing record sets. Read back by **nb 050**, which adds exactly these to each structure's MSA run list. |
| `ADDITIONAL_IML_PATH` | `{"3": [[site, iml], ...], "5": [...]}` - one entry per new IML that needs disaggregation and record selection. Read back by **nb 017**, which merges them into each structure's `additional` field, the site `union` lists and the flat disaggregation grid. |

Re-running nb 017 after this notebook therefore closes the loop; nb 020/021 and 031-036
then pick the new IMLs up as ordinary work.

JSON object keys are strings, so reload the storey count with `int(k)`. A structure appears
twice if both of its tails need fixing, and can appear in both files if one tail is covered
by a reserve and the other is not.

In [9]:
def predict_pc(fc, iml):
    """P[C] at an arbitrary IML, from the lognormal fitted to the analysed stripes."""
    return float(lognorm.cdf(iml, s=fc["dispersion"], scale=fc["median"]))


def iml_at_pc(fc, pc):
    """Inverse of predict_pc: the IML whose fitted P[C] equals pc."""
    return float(lognorm.ppf(pc, s=fc["dispersion"], scale=fc["median"]))


def max_usable_iml(hc):
    """Largest IML with strictly positive MAFE (the truncation ceiling). From nb 017."""
    nonpos = np.where(hc[:, 1] <= 0.0)[0]
    return float(hc[nonpos[0] - 1, 0] if len(nonpos) else hc[-1, 0])


# The IMLs disaggregation has already been run at (nb 017/021): a remediating stripe placed
# on one of these needs new record selection, but no new disaggregation.
DISAGG_GRID = np.loadtxt(DISAGG_GRID_PATH)

with open(HAZARD_CURVE_PATH, "rb") as file:
    hcs = pickle.load(file)

# Per-site hazard limits. The ceiling is the one nb 017 capped its IML choice with, and
# the reason nine 3s structures have no high reserve at all. The disagg grid can always be
# extended downwards, so the floor is just where the hazard curve itself starts.
iml_ceiling = {site: max_usable_iml(hcs[site]["AvgSA"]["mean"]) for site in hcs}
iml_floor = {site: float(hcs[site]["AvgSA"]["mean"][0, 0]) for site in hcs}

print(f"disagg grid ({len(DISAGG_GRID)} IMLs): {np.round(DISAGG_GRID, 3).tolist()}")
print(f"hazard curves span {min(iml_floor.values()):.4f} g to a "
      f"{DISAGG_SIGMA}-sigma ceiling of {min(iml_ceiling.values()):.2f} - "
      f"{max(iml_ceiling.values()):.2f} g")

disagg grid (23 IMLs): [0.164, 0.211, 0.242, 0.27, 0.285, 0.31, 0.33, 0.36, 0.4, 0.45, 0.49, 0.5, 0.55, 0.605, 0.65, 0.714, 0.789, 0.8, 0.89, 0.95, 1.011, 1.15, 1.337]
hazard curves span 0.0005 g to a 4-sigma ceiling of 0.49 - 2.09 g


In [10]:
# Re-read the processed copies rather than reusing anything from section 3, so this section
# stands on its own whether or not the cache was hit above.
diagnostics = []
for site in SITES:
    for ns in N_STOREYS:
        save_fp = processed_path(site, ns)
        if not save_fp.is_file():
            continue
        with open(save_fp, "r") as file:
            fc = json.load(file)

        efc = np.array(fc["efc"])
        # nb 017 writes the reserves as [low, high]; either is null where the trial grid ran
        # out or the neighbour sat above the site's hazard ceiling.
        lo_res, hi_res = stripe_imls[str(site)][structure_tag(site, ns)]["reserves"]
        diagnostics.append({
            "site": site,
            "ns": ns,
            "median": fc["median"],
            "dispersion": fc["dispersion"],
            "pc_min": efc[1, :].min(),
            "pc_max": efc[1, :].max(),
            "needs_low": bool(efc[1, :].min() >= PC_LOWER_THRESHOLD),
            "needs_high": bool(efc[1, :].max() <= PC_UPPER_THRESHOLD),
            "analysed": efc[0, :].tolist(),
            "lo_reserve": lo_res,
            "hi_reserve": hi_res,
            "pc_lo_reserve": predict_pc(fc, lo_res) if lo_res is not None else np.nan,
            "pc_hi_reserve": predict_pc(fc, hi_res) if hi_res is not None else np.nan,
            "iml_ceiling": iml_ceiling[site],
            "iml_floor": iml_floor[site],
        })

diag = pd.DataFrame(diagnostics)
print(f"{len(diag)} structures assessed - {diag.needs_low.sum()} need a lower stripe, "
      f"{diag.needs_high.sum()} need an upper one, "
      f"{(diag.needs_low & diag.needs_high).sum()} need both")
diag.head(10)

114 structures assessed - 28 need a lower stripe, 82 need an upper one, 11 need both


,site,ns,median,dispersion,pc_min,pc_max,needs_low,needs_high,analysed,lo_reserve,hi_reserve,pc_lo_reserve,pc_hi_reserve,iml_ceiling,iml_floor
0,0,3,0.416530,0.249205,0.133333,0.766667,False,True,"[0.31, 0.36, 0.4, 0.5]",0.285,0.55,0.063914,0.867658,0.703768,0.0005
1,0,5,0.374105,0.280749,0.166667,0.600000,False,True,"[0.285, 0.33, 0.36, 0.4]",0.270,0.45,0.122702,0.744706,0.703768,0.0005
2,1,3,0.419410,0.235895,0.066667,0.600000,False,True,"[0.31, 0.36, 0.4, 0.45]",0.285,0.50,0.050727,0.771885,1.453032,0.0005
3,1,5,0.376624,0.170682,0.066667,0.666667,False,True,"[0.285, 0.33, 0.36, 0.4]",0.270,0.45,0.025590,0.851497,1.453032,0.0005
4,2,3,0.411109,0.232067,0.100000,0.666667,False,True,"[0.31, 0.36, 0.4, 0.45]",0.285,0.50,0.057200,0.800528,0.703768,0.0005
5,2,5,0.408853,0.234550,0.066667,0.433333,False,True,"[0.285, 0.33, 0.36, 0.4]",0.270,0.45,0.038441,0.658669,0.703768,0.0005
6,3,3,0.447655,0.250858,0.066667,0.633333,False,True,"[0.31, 0.36, 0.4, 0.5]",0.285,0.55,0.035934,0.794110,0.703768,0.0005
7,3,5,0.431429,0.248289,0.066667,0.400000,False,True,"[0.285, 0.33, 0.36, 0.4]",0.270,0.45,0.029537,0.567391,0.703768,0.0005
8,4,3,0.411874,0.327100,0.233333,0.666667,True,True,"[0.31, 0.36, 0.4, 0.45]",0.285,NaN,0.130139,NaN,0.489786,0.0005
9,4,5,0.361502,0.368948,0.233333,0.600000,True,True,"[0.285, 0.33, 0.36, 0.4]",0.270,0.45,0.214465,0.723585,0.489786,0.0005


### 5.1 Criterion 1 — structures the reserve stripes can rescue

A tail is rescued by its reserve when the reserve exists **and** its predicted P[C] lands
inside the band. A structure is rescued when every tail it needs is either already covered
by an analysed stripe or rescued by a reserve; those reserve stripes go into
`RESERVE_STRIPE_PATH` and can be run against the ensembles already on disk.

Reserves that miss the band are handed to criterion 2, but the *near misses* — a reserve
that overshoots the band yet still crosses the 0.2 / 0.8 threshold — are printed
separately. Those would very probably still yield a usable stripe, and running one is far
cheaper than a fresh disaggregation, so they are worth a look before section 5.2 is acted
on.

In [11]:
def in_band(pc, band):
    return band[0] <= pc <= band[1]


reserve_to_run = {ns: [] for ns in N_STOREYS}   # (site, iml) - existing records, MSA only
unsolved = []                                   # tails that no reserve can cover
near_misses = []                                # reserve outside the band but still useful

for row in diagnostics:
    tails = (
        ("low", row["needs_low"], row["lo_reserve"], row["pc_lo_reserve"],
         PC_BAND_LOW, PC_LOWER_THRESHOLD),
        ("high", row["needs_high"], row["hi_reserve"], row["pc_hi_reserve"],
         PC_BAND_HIGH, PC_UPPER_THRESHOLD),
    )
    for tail, needs, reserve, pc_reserve, band, threshold in tails:
        if not needs:
            continue
        if reserve is not None and in_band(pc_reserve, band):
            reserve_to_run[row["ns"]].append((row["site"], reserve))
            continue
        # Not trustworthy enough to bet the last MSA round on. Flag it anyway if it would
        # still have crossed the pass/fail threshold, then hand the tail to criterion 2.
        if reserve is not None and ((tail == "low" and pc_reserve < threshold)
                                    or (tail == "high" and pc_reserve > threshold)):
            near_misses.append({**row, "tail": tail, "pc_reserve": pc_reserve})
        unsolved.append({**row, "tail": tail, "band": band})

needy = {(r["site"], r["ns"]) for r in diagnostics if r["needs_low"] or r["needs_high"]}
still_bad = {(r["site"], r["ns"]) for r in unsolved}
rescued = needy - still_bad

print("Criterion 1 - can the reserve stripes rescue the curve?")
print(f"  accepting a reserve only if its predicted P[C] is in {PC_BAND_LOW} (low) "
      f"or {PC_BAND_HIGH} (high)")

for ns in N_STOREYS:
    n_total = sum(r["ns"] == ns for r in diagnostics)
    n_needy = sum(n == ns for _, n in needy)
    n_rescued = sum(n == ns for _, n in rescued)
    print(f"\n{ns}s structures - {n_total} assessed")
    print(f"  already bracketed          : {n_total - n_needy}")
    print(f"  rescued by reserve stripes : {n_rescued}")
    print(f"  still need a new IML       : {n_needy - n_rescued}")
    print(f"  reserve stripes to run ({len(reserve_to_run[ns])}):")
    print(f"    {fmt_sites([s for s, _ in reserve_to_run[ns]])}")

print("\nNear misses - reserve outside the band but still past the threshold "
      f"({len(near_misses)}):")
for row in sorted(near_misses, key=lambda r: (r["ns"], r["site"])):
    reserve = row["lo_reserve"] if row["tail"] == "low" else row["hi_reserve"]
    print(f"  site {row['site']:>2} {row['ns']}s  {row['tail']:>4} reserve "
          f"{reserve:.3f} g -> P[C] {row['pc_reserve']:.3f}")

Criterion 1 - can the reserve stripes rescue the curve?
  accepting a reserve only if its predicted P[C] is in (0.067, 0.167) (low) or (0.833, 0.933) (high)

3s structures - 57 assessed
  already bracketed          : 10
  rescued by reserve stripes : 21
  still need a new IML       : 26
  reserve stripes to run (25):
    0, 4, 5, 6, 10, 11, 12, 14, 15, 26, 27, 29, 30, 33, 35, 36, 37, 37, 39, 40, 43, 44, 49, 50, 57

5s structures - 57 assessed
  already bracketed          : 5
  rescued by reserve stripes : 11
  still need a new IML       : 41
  reserve stripes to run (14):
    1, 17, 23, 28, 29, 32, 33, 35, 36, 43, 44, 45, 46, 55

Near misses - reserve outside the band but still past the threshold (26):
  site  2 3s  high reserve 0.500 g -> P[C] 0.801
  site  7 3s  high reserve 0.650 g -> P[C] 0.974
  site  9 3s  high reserve 0.500 g -> P[C] 0.935
  site 13 3s  high reserve 0.500 g -> P[C] 0.829
  site 25 3s  high reserve 0.550 g -> P[C] 0.823
  site 34 3s  high reserve 1.150 g -> P[C] 

### 5.2 Criterion 2 — the minimum set of new IMLs

For every tail no reserve can cover, the fitted lognormal gives the **interval of IMLs**
that would land inside the band, `[iml_at_pc(band_lo), iml_at_pc(band_hi)]`. Each interval
is then resolved in order of cost:

1. **Capped** — the whole interval sits outside the site's hazard curve, in practice
   above its truncation ceiling. The band cannot be reached, so the stripe is pinned to
   the ceiling itself, the highest IML the site can be disaggregated at, and joins the
   pool as a fixed point. It is reported separately because the resulting stripe will
   still fall short of the band. Falling below the bottom of the current disagg grid is
   *not* a cap — that grid can be extended downwards.
2. **Snap to the disagg grid** — if a grid IML falls inside the interval, use the one
   nearest the band midpoint. Disaggregation already exists there, so only record
   selection is needed. IMLs the structure has already run, or already holds as a reserve,
   are excluded: the fitted curve can sit inside the band at an IML whose actual stripe
   came back below the threshold, and re-running that stripe would change nothing.
3. **New IMLs, pooled and minimised** — the remaining intervals from *all* sites and both
   storey counts are pooled and solved as a minimum interval-stabbing problem: sort by
   upper bound, place a point at the upper bound of the first interval not yet hit, drop
   every interval that point covers. The greedy right-endpoint sweep is provably optimal,
   so the result is the fewest distinct IMLs that give every remaining structure a stripe
   inside its band.

In [12]:
def stab_intervals(intervals):
    """Fewest points hitting every interval - greedy sweep on the right endpoints.

    Sorting by upper bound and committing a point at the first uncovered upper bound is
    the classic optimal solution, so len(result) is the true minimum.
    """
    points = []
    for lo, hi in sorted(intervals, key=lambda iv: iv[1]):
        if not points or points[-1] < lo:
            points.append(hi)
    return points


# 1. reachability - the hazard curve is the only hard limit. A band above the truncation
# ceiling cannot be reached at all, so that tail is *capped*: it collapses to the single
# point at the ceiling, the highest IML the site can be disaggregated at, and is carried
# through as a degenerate interval so sites sharing a ceiling share one new IML. A band
# below the bottom of the disagg grid needs no cap - the grid can be extended downwards.
capped, to_place = [], []
for row in unsolved:
    iml_lo, iml_hi = iml_at_pc(row, row["band"][0]), iml_at_pc(row, row["band"][1])
    row = {**row, "iml_lo": iml_lo, "iml_hi": iml_hi}
    if iml_lo > row["iml_ceiling"] or iml_hi < row["iml_floor"]:
        # round to the stored precision, so the pinned IML is the one that ends up in the
        # JSON rather than a value a hair away from it
        limit = round(row["iml_ceiling"] if iml_lo > row["iml_ceiling"]
                      else row["iml_floor"], 4)
        capped.append(row)
        to_place.append({**row, "iml_lo": limit, "iml_hi": limit})
        continue
    to_place.append({**row,
                     "iml_lo": max(iml_lo, row["iml_floor"]),
                     "iml_hi": min(iml_hi, row["iml_ceiling"])})

# 2. grid snap - cheapest of the remaining options, no new disaggregation. A grid IML the
# structure has already got is no remedy at all: its stripe has been run and came back
# below the threshold, and a reserve of its own was already judged in criterion 1. The
# fitted curve can sit above the band there while the stripe itself did not, so without
# this exclusion the notebook would propose re-running a stripe and changing nothing.
grid_hits, remaining = [], []
for row in to_place:
    already = set(row["analysed"]) | {x for x in (row["lo_reserve"], row["hi_reserve"])
                                      if x is not None}
    in_range = DISAGG_GRID[(DISAGG_GRID >= row["iml_lo"] - IML_TOL)
                           & (DISAGG_GRID <= row["iml_hi"] + IML_TOL)]
    cand = np.array([x for x in in_range
                     if not np.any(np.isclose(list(already), x))], dtype=float)
    if len(cand):
        target = iml_at_pc(row, sum(row["band"]) / 2)
        grid_hits.append({**row, "source": "grid",
                          "iml": float(cand[np.argmin(np.abs(cand - target))])})
    else:
        remaining.append(row)

# 3. everything else pooled into one stabbing problem across sites and storey counts
stab_points = stab_intervals([(r["iml_lo"], r["iml_hi"]) for r in remaining])
new_hits = [{**row, "source": "new",
             "iml": next(p for p in stab_points
                         if row["iml_lo"] - IML_TOL <= p <= row["iml_hi"] + IML_TOL)}
            for row in remaining]

new_imls = {ns: [] for ns in N_STOREYS}
for row in grid_hits + new_hits:
    new_imls[row["ns"]].append((row["site"], row["iml"]))


def fmt_covered(rows):
    return ", ".join(f"{r['site']}/{r['ns']}s({r['tail']})"
                     for r in sorted(rows, key=lambda r: (r["ns"], r["site"])))


print("Criterion 2 - stripes that need an IML the reserves cannot provide")
print(f"  {len(unsolved)} unsolved tails: {len(grid_hits)} land on the existing disagg "
      f"grid, {len(remaining)} need a new IML, of which {len(capped)} are pinned to the "
      f"site hazard ceiling")

print(f"\nMINIMUM NUMBER OF IMLs TO DISAGGREGATE: {len(stab_points)}")
for point in stab_points:
    covered = [r for r in new_hits if r["iml"] == point]
    print(f"  {point:.4f} g - {len(covered)} structure(s): {fmt_covered(covered)}")

print(f"\nAlready on the disagg grid, record selection only ({len(grid_hits)}):")
for iml in sorted({r["iml"] for r in grid_hits}):
    covered = [r for r in grid_hits if r["iml"] == iml]
    print(f"  {iml:.4f} g - {len(covered)} structure(s): {fmt_covered(covered)}")

print(f"\nCapped at the hazard ceiling - the band itself is out of reach, so the stripe "
      f"goes as high as the site allows and will still fall short ({len(capped)}):")
for row in sorted(capped, key=lambda r: (r["ns"], r["site"])):
    print(f"  site {row['site']:>2} {row['ns']}s {row['tail']:>4} tail needs "
          f"{row['iml_lo']:.3f}-{row['iml_hi']:.3f} g -> pinned to "
          f"{row['iml_ceiling']:.4f} g, P[C] {predict_pc(row, row['iml_ceiling']):.3f}")

Criterion 2 - stripes that need an IML the reserves cannot provide
  71 unsolved tails: 71 land on the existing disagg grid, 0 need a new IML, of which 7 are pinned to the site hazard ceiling

MINIMUM NUMBER OF IMLs TO DISAGGREGATE: 0

Already on the disagg grid, record selection only (71):
  0.1645 g - 3 structure(s): 12/5s(low), 23/5s(low), 28/5s(low)
  0.2109 g - 6 structure(s): 12/3s(low), 23/3s(low), 17/5s(low), 18/5s(low), 22/5s(low), 27/5s(low)
  0.2423 g - 3 structure(s): 22/3s(low), 28/3s(low), 4/5s(low)
  0.2700 g - 1 structure(s): 58/5s(low)
  0.2850 g - 2 structure(s): 17/3s(low), 58/3s(low)
  0.3300 g - 1 structure(s): 57/5s(low)
  0.3600 g - 1 structure(s): 59/5s(low)
  0.4898 g - 12 structure(s): 4/3s(high), 8/3s(high), 9/3s(high), 19/3s(high), 27/3s(high), 4/5s(high), 5/5s(high), 6/5s(high), 8/5s(high), 14/5s(high), 19/5s(high), 27/5s(high)
  0.5000 g - 4 structure(s): 0/5s(high), 9/5s(high), 20/5s(high), 26/5s(high)
  0.5500 g - 9 structure(s): 1/3s(high), 2/3s(high), 

### 5.3 Write the remediation lists

In [13]:
def dump_stripe_list(stripes, out_fp):
    """{ns: [(site, iml), ...]} -> JSON, IMLs rounded to the 4 dp the IML file uses."""
    payload = {str(ns): [[int(site), round(float(iml), 4)] for site, iml in sorted(pairs)]
               for ns, pairs in stripes.items()}
    with open(out_fp, "w") as file:
        json.dump(payload, file, indent=2)
    return payload


reserve_payload = dump_stripe_list(reserve_to_run, RESERVE_STRIPE_PATH)
new_payload = dump_stripe_list(new_imls, ADDITIONAL_IML_PATH)

# Every reserve stripe must already have a record ensemble, or "no new selection" is a lie.
for ns, pairs in reserve_to_run.items():
    for site, iml in pairs:
        assert np.any(np.isclose(stripe_imls[str(site)]["union"], iml)), \
            f"site {site} {ns}s: no record set selected at {iml} g"

for name, path, payload in (("reserve stripes", RESERVE_STRIPE_PATH, reserve_payload),
                            ("additional IMLs", ADDITIONAL_IML_PATH, new_payload)):
    counts = ", ".join(f"{ns}s {len(v)}" for ns, v in payload.items())
    print(f"{name}: {counts}  ->  {path}")

reserve stripes: 3s 25, 5s 14  ->  C:\Users\clemettn\Documents\phd\data_processed\05_gcim_distributions\reserve_stripes_to_run_AvgSA_03.json
additional IMLs: 3s 26, 5s 45  ->  C:\Users\clemettn\Documents\phd\data_processed\05_gcim_distributions\additional_imls_for_disagg_AvgSA_03.json
